In [1]:
print("Hello, AI Project!")

Hello, AI Project!


In [2]:
import pandas as pd
rfqs = pd.read_csv("rfqs.csv")
parts = pd.read_csv("parts_catalog.csv")
hist = pd.read_csv("historical_rfqs.csv")
rfqs


,rfq_id,customer_name,part_number,quantity,urgency
0,1,Acme Corp,P1001,50,medium
1,2,Beta LLC,P2002,150,high
2,3,Delta Inc,P3003,20,low


In [3]:
rfqs_merged = rfqs.merge(parts, on="part_number", how="left")
rfqs_merged

,rfq_id,customer_name,part_number,quantity,urgency,description,base_price,inventory_level
0,1,Acme Corp,P1001,50,medium,Hex Bolt 3/8,12.50,500
1,2,Beta LLC,P2002,150,high,Steel Bracket,45.00,200
2,3,Delta Inc,P3003,20,low,Lock Washer,2.75,1000


In [4]:
def compute_price(row):
    price = row["base_price"]

    if row["urgency"] == "high":
        price = price * 1.10

    if row["quantity"] > 100:
        price = price * 0.95

    return round(price, 2)

rfqs_merged["quoted_price"] = rfqs_merged.apply(compute_price, axis=1)

rfqs_merged

,rfq_id,customer_name,part_number,quantity,urgency,description,base_price,inventory_level,quoted_price
0,1,Acme Corp,P1001,50,medium,Hex Bolt 3/8,12.50,500,12.50
1,2,Beta LLC,P2002,150,high,Steel Bracket,45.00,200,47.03
2,3,Delta Inc,P3003,20,low,Lock Washer,2.75,1000,2.75


In [5]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
X = hist[["customer_segment","part_category","order_size","lead_time_days"]]
y = hist["won"]
preprocess = ColumnTransformer(transformers=[
("cat", OneHotEncoder(),
["customer_segment","part_category"]),
("num", "passthrough", ["order_size","lead_time_days"])
])
model = Pipeline(steps=[
("preprocess", preprocess),
("model", LogisticRegression())
])
model.fit(X, y)


Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['customer_segment',
                                                   'part_category']),
                                                 ('num', 'passthrough',
                                                  ['order_size',
                                                   'lead_time_days'])])),
                ('model', LogisticRegression())])

In [6]:
rfqs_merged["customer_segment"] = ["industrial","aerospace","automotive"]
rfqs_merged["part_category"] = ["hardware","bracket","washer"]
rfqs_merged["order_size"] = rfqs_merged["quantity"]
rfqs_merged["lead_time_days"] = [7, 14, 3]
X_new = rfqs_merged[[
"customer_segment","part_category",
"order_size","lead_time_days"
]]
rfqs_merged["win_prob"] = model.predict_proba(X_new)[:, 1]
rfqs_merged


,rfq_id,customer_name,part_number,quantity,urgency,description,base_price,inventory_level,quoted_price,customer_segment,part_category,order_size,lead_time_days,win_prob
0,1,Acme Corp,P1001,50,medium,Hex Bolt 3/8,12.50,500,12.50,industrial,hardware,50,7,0.584497
1,2,Beta LLC,P2002,150,high,Steel Bracket,45.00,200,47.03,aerospace,bracket,150,14,0.939971
2,3,Delta Inc,P3003,20,low,Lock Washer,2.75,1000,2.75,automotive,washer,20,3,0.367100


In [7]:
def load_api_key():
    with open("config.txt") as f:
        line = f.read().strip()
    return line.split("=", 1)[1]

api_key = load_api_key()

In [9]:
print(api_key[:6])
print(api_key[-4:])
print(len(api_key))

AIzaSy
r8E8
39


In [10]:
from google import genai

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Write one short sentence saying hello."
)

print(response.text)

Hello, I hope you are having a wonderful day!


In [11]:
from google import genai

client = genai.Client(api_key=api_key)

def generate_email(row):
    prompt = f"""
You are a sales representative.
Write a short, professional quote email.

Customer: {row['customer_name']}
Part: {row['description']}
Quantity: {row['quantity']}
Price: {row['quoted_price']}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )

    return response.text

rfqs_merged["email"] = rfqs_merged.apply(generate_email, axis=1)

rfqs_merged

,rfq_id,customer_name,part_number,quantity,urgency,description,base_price,inventory_level,quoted_price,customer_segment,part_category,order_size,lead_time_days,win_prob,email
0,1,Acme Corp,P1001,50,medium,Hex Bolt 3/8,12.50,500,12.50,industrial,hardware,50,7,0.584497,Subject: Quote: Hex Bolt 3/8 - Acme Corp\n\nDe...
1,2,Beta LLC,P2002,150,high,Steel Bracket,45.00,200,47.03,aerospace,bracket,150,14,0.939971,Subject: Quote: Steel Brackets for Beta LLC\n\...
2,3,Delta Inc,P3003,20,low,Lock Washer,2.75,1000,2.75,automotive,washer,20,3,0.367100,Subject: Quotation: Lock Washers for Delta Inc...


In [12]:
rfqs_merged.to_csv("final_quotes.csv", index=False)